# Weather API: Current Conditions with OpenWeather

**Goal:** call a REST API from Python, handle errors cleanly, and turn the JSON response into a readable weather report.

**Approach:** read the API key from a Windows environment variable (never from the code), look up the city's coordinates with OpenWeather's Geocoding API (which understands U.S. states), request current conditions for those coordinates, check each response for errors, and print the key fields with sunrise and sunset in local time.

**Tools:** Python (requests, datetime), OpenWeather API, JSON

**Context:** master's coursework (DSC540), cleaned up for this portfolio.

**Setup:** get a free key at openweathermap.org, then run this once in PowerShell and restart Jupyter:

`setx OPENWEATHER_API_KEY "your-key-here"`

## 1. Load the API key securely

In [1]:
import os
import datetime as dt
import requests

# the key is read from the environment, so it never appears in the notebook or its output
api_key = os.environ.get('OPENWEATHER_API_KEY')
if not api_key:
    raise RuntimeError('Set the OPENWEATHER_API_KEY environment variable first (see the setup note above).')
print('API key found.')

API key found.


## 2. Find the place, then request its weather, with error handling
The weather endpoint ignores a U.S. state in its city search, so "London, TX" would come back as London, England. OpenWeather's Geocoding API does read the state, so the notebook looks up the coordinates first and then asks for the weather at those coordinates.

In [2]:
def call_openweather(url, params):
    """Call an OpenWeather endpoint; return the JSON, or None after printing a clear message."""
    try:
        response = requests.get(url, params={**params, 'appid': api_key}, timeout=15)
    except requests.RequestException as e:
        print(f'Could not reach OpenWeather: {type(e).__name__}')
        return None
    if response.status_code == 401:
        print('The API key was rejected. Check that it is active.')
        return None
    if response.status_code != 200:
        print(f"Request failed ({response.status_code}): {response.json().get('message', '')}")
        return None
    return response.json()

def get_weather(city, state='', country='US', units='imperial'):
    location = ','.join(part for part in [city, state, country] if part)
    places = call_openweather('https://api.openweathermap.org/geo/1.0/direct', {'q': location, 'limit': 1})
    if places is None:
        return None
    if not places:
        print(f'{location} was not found. Please check the spelling.')
        return None
    place = places[0]
    data = call_openweather('https://api.openweathermap.org/data/2.5/weather',
                            {'lat': place['lat'], 'lon': place['lon'], 'units': units})
    if data is not None:
        # name the place that was looked up (the weather endpoint may name the nearest station instead)
        data['place'] = ', '.join(p for p in [place['name'], place.get('state', ''), place['country']] if p)
    return data

## 3. Turn the JSON into a readable report

In [3]:
def weather_report(data, units='imperial'):
    deg = {'imperial': 'F', 'metric': 'C'}.get(units, 'K')
    speed = 'mph' if units == 'imperial' else 'm/s'
    offset = dt.timedelta(seconds=data['timezone'])           # the city's offset from UTC
    local = lambda ts: (dt.datetime.fromtimestamp(ts, dt.timezone.utc) + offset).strftime('%I:%M %p')
    main = data['main']
    print(f"Weather for {data['place']}")
    print(f"  Conditions:  {data['weather'][0]['description']}")
    print(f"  Temperature: {main['temp']:.0f} {deg} (feels like {main['feels_like']:.0f} {deg}; low {main['temp_min']:.0f}, high {main['temp_max']:.0f})")
    print(f"  Humidity:    {main['humidity']}%   Cloud cover: {data['clouds']['all']}%")
    print(f"  Wind:        {data['wind']['speed']} {speed}   Pressure: {main['pressure']} hPa")
    print(f"  Sunrise:     {local(data['sys']['sunrise'])} local time   Sunset: {local(data['sys']['sunset'])} local time")

## 4. Try it

In [4]:
# two U.S. cities with their states (London, TX shows the state is used), London, England,
# and a made-up city name to show the error handling
for city, state, country in [('Dallas', 'TX', 'US'), ('London', 'TX', 'US'), ('London', '', 'GB'), ('Xyzzyville', 'TX', 'US')]:
    data = get_weather(city, state, country)
    if data:
        weather_report(data)
    print()

Weather for Dallas, Texas, US
  Conditions:  clear sky
  Temperature: 55 F (feels like 54 F; low 53, high 57)
  Humidity:    68%   Cloud cover: 0%
  Wind:        6.91 mph   Pressure: 1021 hPa
  Sunrise:     07:03 AM local time   Sunset: 05:23 PM local time



Weather for London, Texas, US
  Conditions:  clear sky
  Temperature: 48 F (feels like 44 F; low 48, high 54)
  Humidity:    78%   Cloud cover: 0%
  Wind:        9.17 mph   Pressure: 1021 hPa
  Sunrise:     07:10 AM local time   Sunset: 05:38 PM local time



Weather for London, England, GB
  Conditions:  few clouds
  Temperature: 38 F (feels like 35 F; low 36, high 41)
  Humidity:    83%   Cloud cover: 20%
  Wind:        4.61 mph   Pressure: 1020 hPa
  Sunrise:     07:31 AM local time   Sunset: 04:02 PM local time



Xyzzyville,TX,US was not found. Please check the spelling.



## Notes
- **Security:** the key lives only in the `OPENWEATHER_API_KEY` environment variable. The notebook never prints the key or the request URL, which would contain it.
- **Correction from the original version:** the original sent the state as a separate `state_code` parameter, which the API ignores, so "London, TX" actually returned London, England (its July sunrise and sunset times match England). Putting the state in the city search (`city,state,country`) does not help either: the weather endpoint still returns London, England. The notebook now looks up the coordinates with the Geocoding API, which does use the state, and requests the weather for those coordinates.
- **Results change with the weather:** each run shows the conditions at that moment.